In [1]:
import os
import pandas as pd
import re
from datetime import datetime
import gc
import numpy as np


In [2]:
# Path to the current folder containing
folder_path = os.getcwd() 

# Get a list of all CSV files in the folder
file_list = sorted([f for f in os.listdir(folder_path) if f.endswith('.csv')], key=str.casefold)
print("\n".join(file_list))

december-2020-About.csv
december-2020-Complexity_of_the_crisis.csv
december-2020-Conditions_of_people_affected.csv
december-2020-Core_Indicators.csv
december-2020-Country_Indicator_Data.csv
december-2020-Crisis_Indicator_Data.csv
december-2020-Crisis_info.csv
december-2020-Data_Reliability.csv
december-2020-Impact_of_the_crisis.csv
december-2020-Imputed_and_missing_data_hidden.csv
december-2020-Indicator_Date_hidden.csv
december-2020-Indicator_Date_hidden2.csv
december-2020-Indicator_Metadata.csv
december-2020-INFORM_Severity_all_crises.csv
december-2020-INFORM_Severity_country.csv
december-2020-INFORM_Severity_hidden.csv
december-2020-Lists.csv
december-2020-Regional_Crises.csv
december-2020-Reliability.csv
december-2020-Reliability_updated.csv
december-2020-Trends.csv


In [3]:
# Initialize the variables
current_file_index = -1
release_date = None
month = None
year = None
year_month = None

Functions:

In [4]:
import os
import pandas as pd

def load_next_csv(file_list, folder_path, current_file_index):
    """
    Loads the next CSV file from the file list into a DataFrame.
    
    Parameters:
    file_list (list): List of CSV filenames to load.
    folder_path (str): Path to the folder containing the CSV files.
    current_file_index (int): Index of the current file to load.
    
    Returns:
    tuple: DataFrame loaded from the next CSV file, updated file index
    """
    # Check if there are more files to process
    if current_file_index >= len(file_list) - 1:
        print("No more CSV files left to load.")
        return None, current_file_index
    
    # Increment the file index to load the next file
    current_file_index += 1
    
    # Get the next file in the list
    current_file = file_list[current_file_index]
    file_path = os.path.join(folder_path, current_file)
    print(f"Opening file: {current_file}")
    
    # Load the CSV file into a DataFrame without headers for initial inspection
    df = pd.read_csv(file_path, header=None)
    
    return df, current_file_index


In [5]:
def drop_nan(df, column_name):
    """
    Cleans the DataFrame by removing rows where the specified column has NaN,
    removing columns with NaN in the header, and renaming unnamed columns.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to clean.
    column_name (str): The name of the column to check for NaN values in rows.
    
    Returns:
    pd.DataFrame: The cleaned DataFrame.
    """
    # Track rows with NaN in the specified column
    rows_to_delete = df[df[column_name].isna()].index.tolist()

    # Drop rows with NaN in the specified column
    df = df.dropna(subset=[column_name])

    # Track columns with NaN in the header
    columns_to_delete = [col for col in df.columns if pd.isna(col)]
    
    # Replace NaN headers with placeholders (e.g., "Unnamed")
    df.columns = [col if pd.notna(col) else f"Unnamed_{i}" for i, col in enumerate(df.columns)]
    
    # Optionally drop columns that were NaN if not needed
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    # Print deleted rows and columns
    print("Deleted rows (indices):", rows_to_delete)
    print("Deleted columns:", columns_to_delete)

    return df


In [6]:
def clean_dataframe(df, crisis_id_col):
    """
    Cleans the DataFrame by dropping rows where the Crisis ID column has NaN,
    resetting the index, and setting row 0 as the header.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to clean.
    crisis_id_col (str): The name of the Crisis ID column to check for NaN values.
    
    Returns:
    pd.DataFrame: The cleaned DataFrame with updated headers.
    """
    # List to collect rows to drop
    rows_to_drop = []
    cols_to_drop = []
    
    # Identify rows where the Crisis ID column has NaN
    for row in range(len(df)):
        if pd.isna(df.iloc[row, crisis_id_col]):
            rows_to_drop.append(row)
    
    # Drop rows with NaN in the Crisis ID column
    print(f"Dropping NaN rows: {rows_to_drop}")
    df = df.drop(index=rows_to_drop)

    # Reset the index
    df = df.reset_index(drop=True)

    # Identify columns where header is NaN
    for col in range(0, len(df.columns)):
        if pd.isna(df.iloc[0, col]):
            cols_to_drop.append(df.columns[col])

    # Drop columns with NaN in the header
    print(f"Dropping NaN columns: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)

    # Make row 0 the header
    df.columns = df.iloc[0]
    df = df.drop(df.index[0])

    # Rename columns for consistency
    column_mappings = {
        'Crisis Id': 'Crisis Id',
        'Crisisid': 'Crisis Id',
        'CrisisID': 'Crisis Id',
        'ISO3 code': 'ISO3',
        'Iso3 Code': 'ISO3',
        'Iso3': 'ISO3',
        'Iso 3': 'ISO3',
        # Add any other variations that need standardization
    }
    df = df.rename(columns=column_mappings)

    # Standardize target columns to titlecase strings and strip whitespace
    target_columns = ['Crisis', 'Drivers', 'Crisis Id', 'Country', 'ISO3']
    for col in target_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.title().str.strip()

    df.columns = df.columns.str.title()

    return df


In [35]:
def get_dataframe_stats(df):
    """
    Prints the number of rows and columns in the DataFrame.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to analyze.
    """
    num_rows, num_columns = df.shape
    return f"{num_rows} rows x {num_columns} columns"

In [8]:
#####-MERGE-#####


def merge_df(df_1, df_2, on_col, merge_type='outer'):
    # Add suffixes to all columns except the key column
    df_1 = df_1.rename(columns={col: f"{col}[df1]" for col in df_1.columns if col != on_col})
    df_2 = df_2.rename(columns={col: f"{col}[df2]" for col in df_2.columns if col != on_col})
    
    # Merge the DataFrames
    merged_df = pd.merge(df_1, df_2, on=on_col, how=merge_type)

    # Remove suffixes by renaming columns
    merged_df.columns = [col.replace('[df1]', '').replace('[df2]', '') for col in merged_df.columns]
    
    # Identify duplicated columns (keeping only the first occurrence)
    duplicate_columns = [col for col in merged_df.columns if merged_df.columns.tolist().count(col) > 1]

    # Remove duplicate columns, keeping only the first occurrence
    merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

    # Identify rows where all values are duplicates
    duplicate_rows = merged_df.duplicated(keep=False)  # Marks all rows that are full duplicates

    # Get the indices of duplicate rows
    duplicate_indices = merged_df[duplicate_rows].index.tolist()

    # Count the number of duplicate rows
    num_deleted_rows = len(duplicate_indices)

    # Drop these duplicate rows from the DataFrame
    merged_df_cleaned = merged_df[~duplicate_rows].reset_index(drop=True)

    # Print the number of deleted rows and the row indices
    print(f"Number of rows deleted: {num_deleted_rows}")
    print("Indices of deleted duplicate rows:", duplicate_indices)
    print("DataFrame after removing fully duplicate rows:")
    print(merged_df_cleaned)

    return merged_df_cleaned



About.csv

In [9]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: december-2020-About.csv
Current file index is: 0


,0
0,NaN
1,release:
2,2020-12-31 00:00:00
3,INFORM SEVERITY INDEX
4,December 2020
5,OBJECTIVES AND PROCESS\nThe INFORM Severity In...
6,ANALYTICAL FRAMEWORK AND METHODOLOGY\nThe INFO...
7,NaN
8,"DATA SOURCES, USE AND LIMITATIONS\nThe INFORM ..."
9,FURTHER INFORMATION\nThe development of the IN...


In [10]:
# Function to check if a string is a date in different formats
def is_date(string):
    formats = ["%d/%m/%Y", "%Y-%m-%d %H:%M:%S"]
    for fmt in formats:
        try:
            return datetime.strptime(string, fmt).strftime("%d/%m/%Y")
        except ValueError:
            continue
    return False

In [11]:
current_file = file_list[current_file_index]
print(f"Current file: {current_file}")
if current_file.endswith("About.csv"):
    # Identify the first column by its index position
    first_column = df.columns[0]
    
    # Search for the word 'release' in the first column
    for i, row in df.iterrows():
        if str(row[first_column]).strip().lower() == "release:":
            # Check if the next row exists and if it contains a date
            next_row_value = df.at[i + 1, first_column] if i + 1 < len(df) else None
            formatted_date = is_date(str(next_row_value))
            if formatted_date:
                release_date = formatted_date
                day, month, year = release_date.split("/")  # Split the date into parts
                month = int(month)
                year = int(year)
                year_month = f"{year}_{month:02d}"
            break

Current file: december-2020-About.csv


In [12]:
# Display the extracted release date, month, year, and year_month
if release_date:
    print(f"Release date for {current_file}: {release_date}")
    print(f"Month: {month}")
    print(f"Year: {year}")
    print(f"Year-Month: {year_month}")
else:
    print("\n" + f"No release date found in {current_file}.")

Release date for december-2020-About.csv: 31/12/2020
Month: 12
Year: 2020
Year-Month: 2020_12


In [13]:
pd.set_option('display.max_columns', None)


df1. Complexity_of_the_crisis.csv

In [14]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: december-2020-Complexity_of_the_crisis.csv
Current file index is: 1


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,Empowerment,BTI - Democracy Status,Trust in society,Ethnic Fractionalisation,size of excluded ethnic groups,Ethnic fractionalisation,Gender Inequality,Income Gini coefficient,Inequality,Social cohesion,Conflict Intensity,Total killed in all crisis,Safety and security,Corruption Perception,Rule of Law (WGI),Rule of Law (BTI),Freedom in the World,Rule of Law,Society and safety,# of different types of affected population gr...,Diversity of groups affected,Impediments to entry into country (bureaucrati...,Restriction of movement (impediments to freedo...,Interference into implementation of humanitari...,"Violence against personnel, facilities and assets",Access of Humanitarian Actors to Affected Popu...,Denial of existence of humanitarian needs or e...,Restriction and obstruction of access to servi...,Access of People in need to Aid,Ongoing insecurity/hostilities affecting human...,Presence of mines and improvised explosive dev...,Physical constraints in the environment (obsta...,Physical and Security Constraints,Humanitarian access,Operating environment
2,NaN,NaN,NaN,NaN,MAX,14,10,NaN,1,0.5,NaN,0.75,65,NaN,NaN,NaN,3000,NaN,100,2.5,10,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,MIN,0,0,NaN,0,0,NaN,0,25,NaN,NaN,NaN,1,NaN,0,-2.5,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CRISIS,TYPE,CRISIS ID,COUNTRY,Iso3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,Complex crisis in Zimbabwe,Complex crisis,ZWE001,Zimbabwe,ZWE,5,2.8,3.9,1.5,0.3,0.9,3.5,2.4,2.95,2.5833333333333335,3,1.7,2.35,3.8,3.8,3.4,3.6,3.7,2.9,5,5,1,2,0,0,2,0,2,2,0,2,0,2,2,3.5
153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Complexity_of_the_crisis.csv"):
    if df.iloc[1, 5] == "Empowerment" and pd.isna(df.iloc[4, 5]):
        # Copy the content from row 1 (columns 5-39) down into row 4 (columns 5-39)
        df.iloc[4, 5:df.shape[1]] = df.iloc[1, 5:df.shape[1]]
    # Delete empty rows
    df = clean_dataframe(df, crisis_id_col = 2)
    df1 = df
else: print("Wrong file")

Dropping NaN rows: [0, 1, 2, 3, 153, 154, 155, 156]
Dropping NaN columns: []


In [16]:
# Display the updated DataFrame
df1

,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Ethnic Fractionalisation,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment
1,Complex crisis in Afghanistan,Complex crisis,AFG001,Afghanistan,Afg,3.6,3.4,3.5,3.7,0,1.85,3.8,x,3.8,3.0499999999999994,5,5,5,4.2,4.2,3.5,3.7,3.9,4,5,5,1,3,2,3,4,0,3,3,3,3,3,5,4,4.5
2,Mixed migration flows in Algeria,International displacement,DZA002,Algeria,Dza,3.9,2.7,3.3,2,2.7,2.35,3,0.3,1.65,2.4333333333333336,3,4,3.5,3.3,3.3,2.9,3.3,3.2,3,2,2,1,0,1,0,2,0,2,2,0,1,0,1,2,2
3,Sahrawi refugees in Algeria,International displacement,DZA003,Algeria,Dza,3.9,2.7,3.3,2,2.7,2.35,3,0.3,1.65,2.4333333333333336,3,4,3.5,3.3,3.3,2.9,3.3,3.2,3,1,1,1,1,2,0,2,0,0,0,0,1,2,3,2,1.5
4,Multiple crises in Algeria,Multiple crises country,DZA004,Algeria,Dza,3.9,2.7,3.3,2,2.7,2.35,3,0.3,1.65,2.4333333333333336,3,4,3.5,3.3,3.3,2.9,3.3,3.2,3,2,2,1,1,2,0,2,0,2,2,0,1,1,2,2,2
5,Western Mediterranean Route,Regional crisis,REG008,"Algeria, Morocco, Spain","Dza, Mar, Esp",2.7,2.9,2.8,2.3,3.2,2.75,2.2,1.1,1.6500000000000001,2.4,3,3.2,3.1,2.7,2.5,3.1,2.3,2.7,2.7,2,2,1,0,2,0,2,0,2,2,0,1,0,1,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,Floods in central Vietnam,Flood,VNM002,Vietnam,Vnm,4.6,3.2,3.9,1.4,1,1.2,2.1,1.3,1.7000000000000002,2.2666666666666666,5,3.4,4.2,3.2,2.5,3.5,4,3.3,3.3,2,2,0,0,0,0,0,0,0,0,2,0,3,4,1,1.5
145,Conflict in Yemen,Complex crisis,YEM001,Yemen,Yem,4.3,4.3,4.3,3.1,0,1.55,5,1.5,3.25,3.033333333333333,3,5,4,4.3,4.3,4.4,4.5,4.4,3.8,5,5,2,3,3,3,5,2,3,5,3,3,3,5,5,5
146,Mixed migration flows in Yemen,International displacement,YEM002,Yemen,Yem,4.3,4.3,4.3,3.1,0,1.55,5,1.5,3.25,3.033333333333333,3,5,4,4.3,4.3,4.4,4.5,4.4,3.8,2,2,2,3,3,3,5,2,3,5,3,3,3,5,5,3.5
147,Drought in Zambia,Drought,ZMB002,Zambia,Zmb,2.1,2.1,2.1,3.5,0,1.75,3.6,4,3.8,2.5500000000000003,0,x,0,3.3,3,2.9,2.3,2.9,1.8,3,3,0,0,0,0,0,0,0,0,0,0,1,1,0,1.5


df2. Conditions_of_people_affected.csv

In [17]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: december-2020-Conditions_of_people_affected.csv
Current file index is: 2


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,# of people in none/minimal conditions - Level 1,# of people in stressed conditions - level 2,# of people in moderate conditions - level 3,# of people severe conditions - level 4,# of people extreme conditions - level 5,People in Need,% of people in none/minimal conditions - Level 1,% of people in stressed conditions - level 2,% of people in moderate conditions - level 3,% of people severe conditions - level 4,% of people extreme conditions - level 5,Concentration of conditions,Conditions of people affected
2,NaN,NaN,NaN,NaN,MAX,NaN,NaN,NaN,NaN,NaN,7,0.05,0.05,0.05,0.05,0.05,NaN,NaN
3,NaN,NaN,NaN,NaN,MIN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CRISIS,TYPE OF CRISIS,CRISIS ID,COUNTRY,Iso3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,Drought in Zambia,Drought,ZMB002,Zambia,ZMB,2.526,2.387,1.651,0.033,0,3.7,0.3666715052983016,0.3464944113804616,0.23965742488024386,0.004790245318623893,0,3,3.35
152,Complex crisis in Zimbabwe,Complex crisis,ZWE001,Zimbabwe,ZWE,7.645,0,7,0,0,4.7,0.5220211676340047,0,0.4779788323659952,0,0,3,3.85
153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Conditions_of_people_affected.csv"):
    if df.iloc[1, 5] == "# of people in none/minimal conditions - Level 1" and pd.isna(df.iloc[4, 5]):
        # Copy the content from row 1 (columns 5-39) down into row 4 (columns 5-39)
        df.iloc[4, 5:df.shape[1]] = df.iloc[1, 5:df.shape[1]]
    # Delete empty rows
    df = clean_dataframe(df, crisis_id_col = 2)
    df2 = df
else: print("Wrong file")

Dropping NaN rows: [0, 1, 2, 3, 153, 154, 155]
Dropping NaN columns: []


In [19]:
# Display the updated DataFrame
df2

,Crisis,Type Of Crisis,Crisis Id,Country,Iso3,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected
1,Complex crisis in Afghanistan,Complex crisis,AFG001,Afghanistan,Afg,0,21.7,10.447,3.553,0,5,0,0.6078431372549019,0.2926330532212885,0.09952380952380953,0,4,4.5
2,Mixed migration flows in Algeria,International displacement,DZA002,Algeria,Dza,x,x,x,x,x,x,x,x,x,x,x,x,x
3,Sahrawi refugees in Algeria,International displacement,DZA003,Algeria,Dza,0.049,0.084,0.09,0,0,1.6,0.21973094170403587,0.37668161434977576,0.40358744394618834,0,0,3,2.3
4,Multiple crises in Algeria,Multiple crises country,DZA004,Algeria,Dza,x,x,x,x,x,x,x,x,x,x,x,x,x
5,Western Mediterranean Route,Regional crisis,REG008,"Algeria, Morocco, Spain","Dza, Mar, Esp",x,x,x,x,x,x,x,x,x,x,x,x,x
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,Floods in central Vietnam,Flood,VNM002,Vietnam,Vnm,6.2,1.323,0.177,0,0,2.1,0.8051948051948052,0.17181818181818181,0.022987012987012986,0,0,2,2.05
145,Conflict in Yemen,Complex crisis,YEM001,Yemen,Yem,0,6.337,9.8,14.3,0.064,5,0,0.2077704918032787,0.32131147540983607,0.46885245901639344,0.002098360655737705,4,4.5
146,Mixed migration flows in Yemen,International displacement,YEM002,Yemen,Yem,3.873,0,0.245,0.177,0,2.7,0.9508961453474097,0,0.06015222194942303,0.04345691136754235,0,3,2.85
147,Drought in Zambia,Drought,ZMB002,Zambia,Zmb,2.526,2.387,1.651,0.033,0,3.7,0.3666715052983016,0.3464944113804616,0.23965742488024386,0.004790245318623893,0,3,3.35


In [20]:
#####-MERGE-#####

merged_df = merge_df(df1, df2, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis                        Type  \
0              Complex crisis in Afghanistan              Complex crisis   
1       Nagorno-Karabakh Conflict in Armenia                    Conflict   
2    Nagorno-Karabakh conflict in Azerbaijan                    Conflict   
3                         Complex in Burundi              Complex crisis   
4                   Conflict in Burkina Faso                    Conflict   
..                                       ...                         ...   
143                Cyclone Harold in Vanuatu            Tropical cyclone   
144                        Conflict in Yemen              Complex crisis   
145           Mixed migration flows in Yemen  International displacement   
146                        Drought in Zambia                     Drought   
147               Complex crisis in Zimbabwe         

,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected
0,Complex crisis in Afghanistan,Complex crisis,AFG001,Afghanistan,Afg,3.6,3.4,3.5,3.7,0,3.8,x,3.8,3.0499999999999994,5,5,5,4.2,4.2,3.5,3.7,3.9,4,5,5,1,3,2,3,4,0,3,3,3,3,3,5,4,4.5,Complex crisis,0,21.7,10.447,3.553,0,5,0,0.6078431372549019,0.2926330532212885,0.09952380952380953,0,4,4.5
1,Nagorno-Karabakh Conflict in Armenia,Conflict,ARM002,Armenia,Arm,2.9,1.5,2.2,0.2,0.2,1.7,1.2,1.45,1.2833333333333334,3,0.4,1.7,2.9,2.6,2,2.4,2.5,1.8,3,3,0,0,0,0,0,0,0,0,0,2,0,2,1,2,Conflict,x,x,x,x,x,x,x,x,x,x,x,x,x
2,Nagorno-Karabakh conflict in Azerbaijan,Conflict,AZE002,Azerbaijan,Aze,3.6,3.3,3.45,0.8,0.6,2.1,x,2.1,2.0833333333333335,3,2.8,2.9,3.5,3.1,3.5,4.5,3.7,2.9,3,3,2,2,0,0,2,0,2,2,3,3,0,4,3,3,Conflict,x,x,x,x,x,x,x,x,x,x,x,x,x
3,Complex in Burundi,Complex crisis,BDI001,Burundi,Bdi,2.1,3.2,2.6500000000000004,1.3,0,3.5,1.7,2.6,1.9666666666666668,3,3.4,3.2,4.1,3.9,3.8,4.4,4.1,3.1,5,5,1,2,2,0,2,2,0,2,1,0,2,3,2,3.5,Complex crisis,6.368,3.955,1.206,0.128,0,3.5,0.5462812044265248,0.33928111864115984,0.10345715021017414,0.010980526722141202,0,3,3.25
4,Conflict in Burkina Faso,Conflict,BFA002,Burkina Faso,Bfa,1.1,1.9,1.5,2.8,0,4.1,1.3,2.6999999999999997,1.8666666666666665,3,5,4,3,2.9,2.9,2.2,2.8,2.9,4,4,0,2,2,3,3,0,2,2,3,0,3,4,3,3.5,Conflict,3.459,3.107,0.49,0.764,0.916,3.9,0.3959478021978022,0.3556547619047619,0.05608974358974359,0.08745421245421245,0.10485347985347986,5,4.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,Cyclone Harold in Vanuatu,Tropical cyclone,VUT002,Vanuatu,Vut,0.7,x,0.7,0,0,x,1.6,1.6,0.7666666666666666,0,0.9,0.45,2.7,2.3,x,0.9,2,1.1,1,1,1,0,0,0,1,0,0,0,0,0,3,3,1,1,Tropical cyclone,0.075,0.033,0.123,0,0,1.8,0.3246753246753247,0.14285714285714285,0.5324675324675324,0,0,3,2.4
144,Conflict in Yemen,Complex crisis,YEM001,Yemen,Yem,4.3,4.3,4.3,3.1,0,5,1.5,3.25,3.033333333333333,3,5,4,4.3,4.3,4.4,4.5,4.4,3.8,5,5,2,3,3,3,5,2,3,5,3,3,3,5,5,5,Complex crisis,0,6.337,9.8,14.3,0.064,5,0,0.2077704918032787,0.32131147540983607,0.46885245901639344,0.002098360655737705,4,4.5
145,Mixed migration flows in

df3. Core_Indicators.csv

In [21]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df


Opening file: december-2020-Core_Indicators.csv
Current file index is: 3


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252
0,Crisis,Type of crisis,Crisis ID,Country,ISO3 code,Total population,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Landmass affected,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People exposed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People affected,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People displaced,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Injuries reported,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Illness cases reported,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fatalities reported,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Buildings damaged,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Buildings destroyed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Buildings in the affected area,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Economic Losses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People in Need,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Minimal humanitarian conditions - Level 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stressed humanitarian conditions - Level 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Moderate humanitarian conditions - Level 3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Severe humanitarian conditions - Level 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Extreme humanitarian conditions - Level 5,NaN,NaN,NaN,NaN,NaN,NaN,Fatalities in all crises,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Crisis affected groups,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People facing limited access constraints,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People facing restricted access constraints,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Impediments to entry into country (bureaucrati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Restriction of movement (impediments to freedo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Interference into implementation of humanitari...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Violence against personnel, facilities and assets",NaN,NaN,NaN,NaN,NaN,NaN,NaN,Denial of existence of humanitarian needs or e...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Restriction and obstruction of access to servi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ongoing insecurity/hostilities affecting human...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Presence of mines and improvised explosive dev...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Physical constraints in the environment (obsta...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,NaN,0,NaN,0,Helper 0,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 1,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 2,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 3,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 4,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 5,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 6,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 7,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 8,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 9,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 10,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 11,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 12,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 13,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 14,Figure,Source,Reliability,Rel

In [22]:
def find_column_name(df, row, col):
    # Start one row up from the "Helper" cell
    current_row = row - 1
    current_col = col  # Starting column

    # Iterate to the right until a non-empty cell is found
    while current_col < df.shape[1]:  # Ensure we don't go beyond the last column
        if pd.notna(df.iloc[current_row, current_col]):
            return df.iloc[current_row, current_col]
        current_col += 1  # Move to the next column

    # If no column name is found, return a placeholder or None
    return None

# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Core_Indicators.csv"):
    for col in range(5, len(df.columns)):
        if pd.notna(df.loc[1, col]) and re.match(r"^Helper.*", str(df.loc[1, col])):
        # Try to find the column name by searching upwards
            col_name = find_column_name(df, 1, col)
        df.loc[0, col] =f"{col_name} [{str(df.loc[1, col])}]"
    df = clean_dataframe(df, crisis_id_col = 3)
    
    df3 = df
else: print("Wrong file")

Dropping NaN rows: [1, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213]
Dropping NaN columns: []


In [23]:
df3

,Crisis,Type Of Crisis,Crisis Id,Country,Iso3,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildings Destroyed [Source],Buildings Destroyed [Reliability],Buildings Destroyed [Reliability Score],Buildings Destroyed [Justification],Buildings Destroyed [Link],Buildings Destroyed [Date],Buildings In The Affected Area [Helper 10],Buildings In The Affected Area [Figure],Buildings In The Affected Area [Source],Buildings In The Affected Area [Reliability],Buildings In The Affected Area [Reliability Score],Buildings In The Affected Area [Justification],Buildings In The Affected Area [Link],Buildings In The Affected Area [Date],Economic Losses [Helper 11],Economic Losses [Figure],Economic Losses [Source],Economic Losses [Reliability],Economic Losses [Reliability Score],Economic Losses [Justification],Economic Losses [Link],Economic Losses [Date],People In Need [Helper 12],People In Need [Figure],People In Need [Source],People In Need [Reliability],People In Need [Reliability Score],People In Need [Justification],People In Need [Link],People In Need [Date],Minimal Humanitarian Conditions - Level 1 [Helper 13],Minimal Humanitarian Conditions - Level 1 [Figure],Minimal Humanitarian Conditions - Level 1 [Source],Minimal Humanitarian Conditions - Level 1 [Reliability],Minimal Humanitarian Conditions - Level 1 [Reliability Score],Minimal Humanitarian Conditions - Level 1 [Justification],Minimal Humanitarian Conditions - Level 1 [Link],Minimal Humanitarian Conditions - Level 1 [Date],Stressed Humanitarian Conditions - Level 2 [Helper 14],Stressed Humanitarian Conditions - Level 2 [Figure],Stressed Humanitarian Conditions - Level 2 [Source],Stressed Humanitarian Conditions - Level 2 [Reliability],Stressed Humanitarian Conditions - Level 2 [Reliability Score],Stressed Humanitarian Conditions - Level 2 [Justification],Stressed Humanitarian 

In [24]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df3, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis                        Type  \
0              Complex crisis in Afghanistan              Complex crisis   
1       Nagorno-Karabakh Conflict in Armenia                    Conflict   
2    Nagorno-Karabakh conflict in Azerbaijan                    Conflict   
3                         Complex in Burundi              Complex crisis   
4                   Conflict in Burkina Faso                    Conflict   
..                                       ...                         ...   
143                Cyclone Harold in Vanuatu            Tropical cyclone   
144                        Conflict in Yemen              Complex crisis   
145           Mixed migration flows in Yemen  International displacement   
146                        Drought in Zambia                     Drought   
147               Complex crisis in Zimbabwe         

,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

df4. Country_Indicator_Data.csv

In [25]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: december-2020-Country_Indicator_Data.csv
Current file index is: 4


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,CRISIS,ISO3,Ethnic Fractionalisation Index,Empowerment Rights Index,size of excluded ethnic groups,BTI - Democracy Status,Gender Inequality Index,Income Gini coefficient,Conflict Intensity (HIIK),People killed in all crises,Rule of Law (WGI),Rule of Law (BTI),Freedom in the World Index,CPI,Total Population,Land area (sq. km)
1,Year,NaN,2017,2011,2004-2006,2020,2018,2006-2018,2019,2020,2019,2020,2020,2019,2020,NaN
2,Unit of Measurament,NaN,Index,Index,%,Index,Index,Index,Index,Number,Index,Index,Index,Index,Number,sq. Km
3,Afghanistan,AFG,0.7497860033955295,4,0,3.2833333333333337,0.5747417060582192,x,5,11393,-1.713526964187622,3,27,16,35700000,652860
4,Albania,ALB,0.32080001203417774,9,0.04,7.15,0.23407785371782353,33.2000007629395,3,x,-0.41117939352989197,5.5,67,35,x,27400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
352,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
353,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
354,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Country_Indicator_Data.csv"):
    cols_to_drop = []
    for col in range(2, len(df.columns)):
        if pd.isna(df.iloc[0, col]):
            cols_to_drop.append(df.columns[col])
            df[col] = df[col].astype(object)  # Convert the entire column to object type
        df.iloc[0, col] =f"{df.iloc[0, col]} [{str(df.iloc[1, col])}] [{df.iloc[2, col]}]"

    print(f"Columns to drop: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)

    df = clean_dataframe(df, crisis_id_col = 1)
    df4 = df
else: print("Wrong file")

Columns to drop: []
Dropping NaN rows: [1, 2, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355]
Dropping NaN columns: []


In [27]:
df4

,Crisis,Iso3,Ethnic Fractionalisation Index [2017] [Index],Empowerment Rights Index [2011] [Index],Size Of Excluded Ethnic Groups [2004-2006] [%],Bti - Democracy Status [2020] [Index],Gender Inequality Index [2018] [Index],Income Gini Coefficient [2006-2018] [Index],Conflict Intensity (Hiik) [2019] [Index],People Killed In All Crises [2020] [Number],Rule Of Law (Wgi) [2019] [Index],Rule Of Law (Bti) [2020] [Index],Freedom In The World Index [2020] [Index],Cpi [2019] [Index],Total Population [2020] [Number],Land Area (Sq. Km) [Nan] [Sq. Km]
1,Afghanistan,Afg,0.7497860033955295,4,0,3.2833333333333337,0.5747417060582192,x,5,11393,-1.713526964187622,3,27,16,35700000,652860
2,Albania,Alb,0.32080001203417774,9,0.04,7.15,0.23407785371782353,33.2000007629395,3,x,-0.41117939352989197,5.5,67,35,x,27400
3,Algeria,Dza,0.4031999581336967,3,0.27,4.7,0.4430680424365795,27.6000003814697,3,651,-0.8154637813568115,4.25,34,35,43100000,2381741
4,Angola,Ago,0.7626000091969966,4,0.62,4.65,0.5779989037523888,51.2999992370605,3,x,-1.0543431043624878,3.75,32,26,x,1246700
5,Antigua and Barbuda,Atg,0,13,0,x,x,x,0,x,0.4052046537399292,x,85,x,x,440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,Venezuela,Ven,0.26454201694521295,7,0.12000000000000001,3.083333333333333,0.45795582144607083,46.9000015258789,3,8253,-2.3200535774230957,1.75,16,16,27365000,882050
188,Viet Nam,Vnm,0.27606595953145563,1,0.101,3.5666666666666664,0.3138494002746719,35.7000007629395,5,243,-0.016898753121495247,3,20,37,96484000,310070
189,Yemen,Yem,0.6249999926611779,2,0,1.5,0.8341737424646883,36.7000007629395,3,12222,-1.7733464241027832,1.25,11,15,30500000,527970
190,Zambia,Zmb,0.7025999877020714,8,0,5.75,0.5403252994048439,57.0999984741211,0,x,-0.4620692729949951,4.25,54,34,17885000,743390


In [ ]:
#####-MERGE-#####
merged_df = merge_df(merged_df, df4, on_col="Crisis")
merged_df

KeyError: 'Country'

df5. Crisis_Indicator_Data.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Crisis_Indicator_Data.csv"):
    for col in range(5, len(df.columns)):
        if pd.notna(df.loc[1, col]):
            df.loc[0, col] =f"{df.loc[0, col]} [{df.loc[1, col]}]" 
    df = clean_dataframe(df, crisis_id_col = 2)
    df5 = df
else: print("Wrong file")

In [ ]:
df5

In [ ]:
merged_df = merge_df(merged_df, df5, on_col="Crisis Id")
merged_df

df6. Crisis_info.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Crisis_info.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df6 = df
else: print("Wrong file")

In [ ]:
df6

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df6, on_col="Crisis Id")
merged_df

df7. Data_Reliability.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Data_Reliability.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df7 = df
else: print("Wrong file")

In [ ]:
df7['Crisis Id'] = [id.upper() for id in df7['Crisis Id']]
df7

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df7, on_col="Crisis Id")
merged_df

df8. Impact_of_the_crisis.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Impact_of_the_crisis.csv"):
    if df.iloc[1, 5] == "Area affected - absolute" and pd.isna(df.iloc[4, 5]):
        # Copy the content from row 1 (columns 5-...) down into row 4 (columns 5-...)
        df.iloc[4, 5:df.shape[1]] = df.iloc[1, 5:df.shape[1]]
    df = clean_dataframe(df, crisis_id_col = 0)
    df8 = df
else: print("Wrong file")

In [ ]:
df8

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df8, on_col="Crisis Id")
merged_df

Imputed_and_missing_data_hidden.csv -- No data

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Indicator_Date_hidden.csv -- No data

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Indicator_Date_hidden2.csv -- No data

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Indicator_Metadata.csv -- No data, just description of features

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


In [ ]:
df = clean_dataframe(df, crisis_id_col = 0)
df.head()

df9. INFORM_Severity___all_crises.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("all_crises.csv"):
    df = clean_dataframe(df, crisis_id_col = 1)
    df9 = df
else: print("Wrong file")

In [ ]:
df9

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df9, on_col="Crisis Id")
merged_df

df10. INFORM_Severity___country.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("country.csv"):
    df = clean_dataframe(df, crisis_id_col = 1)
    df10 = df
else: print("Wrong file")

In [ ]:
df10

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df10, on_col="Crisis Id")
merged_df

INFORM_Severity___hidden.csv -- Was hidden. No Data to be used

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

df11. Lists.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Lists.csv"):
    df = clean_dataframe(df, crisis_id_col = 2)
    df11 = df
else: print("Wrong file")

In [ ]:
df11

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df11, on_col="Crisis Id")
merged_df

df12. Log.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Log.csv"):
    df = clean_dataframe(df, crisis_id_col = 1)
    df12 = df
else: 
    print("Wrong file")
    current_file_index -= 1

In [ ]:
# df12

In [ ]:
#####-MERGE-#####

# merged_df = merge_df(merged_df, df12, on_col="Crisis Id")
merged_df

df13. Regional_Crises.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Regional_Crises.csv"):
    for col in range(5, len(df.columns)):
        if pd.notna(df.loc[1, col]):
            df.loc[0, col] =f"{df.loc[0, col]} [{df.loc[1, col]}]" 
    df = clean_dataframe(df, crisis_id_col = 1)
    df13 = df
else: print("Wrong file")

In [ ]:
df13

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df13, on_col="Crisis Id")
merged_df

df14. Reliability.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Reliability.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df14 = df
else: print("Wrong file")

In [ ]:
df14

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df14, on_col="Crisis Id")
merged_df

df15. Reliability_updated.csv.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Reliability_updated.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df15 = df
else: print("Wrong file")


In [ ]:
df15

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df15, on_col="Crisis Id")
merged_df

df16. Trends.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Trends.csv"):
    df = clean_dataframe(df, crisis_id_col = 2)
    df16 = df
else: print("Wrong file")


In [ ]:
df16

In [ ]:
#####-MERGE-#####

# merged_df = merge_df(merged_df, df16, on_col="Crisis Id")
merged_df

In [ ]:
# Remove rows where there are NaNs in the Crisis Id column and headers with NaNs or Unnamed
# merged_df =  drop_nan(merged_df, column_name="Crisis Id")

In [ ]:
# Add the year_month column as the first column
merged_df.insert(0, 'YYYY_MM', year_month)

# Reorder columns to have Crisis ID as the second column
columns = ['YYYY_MM', 'Crisis Id'] + [col for col in merged_df.columns if col not in ['YYYY_MM', 'Crisis Id']]
merged_df = merged_df[columns]

merged_df


In [ ]:
merged_df.to_csv(f"{year_month}_merged.csv", index=False)

In [ ]:
print(year_month)

In [ ]:
final_df = pd.read_csv(f"{year_month}_merged.csv")
final_df.shape

In [ ]:
print(final_df.name)